<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/15_deep_learning_on_unseen_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 15: Deep Learning on Unseen Data - Introducing Federated Learning**

This notebook summarizes Chapter 15 of *Grokking Deep Learning*:

1. The Problem of Privacy in Deep Learning
2. Federated Learning
3. Learning to Detect Spam
4. Let's Make It Federated
5. Hacking into Federated Learning
6. Secure Aggregation
7. Homomorphic Encryption
8. Homomorphically Encrypted Federated Learning
9. Summary

This chapter asks what can be done when training data is too personal or too sensitive to be collected in one place. It introduces **federated learning**, in which the model travels to the data instead of the data travelling to the model, and it illustrates the idea with a small spam classifier trained on the Enron email corpus. The chapter then shows that even a single weight update can leak the content of an email, and it protects the updates with **secure aggregation** based on **homomorphic encryption**. The code reuses the deep learning framework that was built in Chapter 13 (in particular its embedding class).

The chapter opens with a remark by Stephen King (*Hearts in Atlantis*, 1999): friends do not spy on one another, because true friendship also means respecting privacy.

---
## **1. The Problem of Privacy in Deep Learning**

**Deep learning, and the tools built for it, usually take for granted that you have access to your training data.**

### **1.1 Valuable Models Need Personal Data**

Deep learning is a subfield of machine learning, and machine learning is about learning from data. Very often, however, that data is deeply personal. The most meaningful models are the ones that interact with the most personal information about human lives, and they can tell us things about ourselves that would otherwise be hard to find out. In the author's words, a deep learning model can study thousands of lives in order to help you understand your own.

The primary natural resource of deep learning is **training data**, whether it is synthetic or natural. Without it, nothing can be learned. Since the most valuable use cases are usually tied to the most personal datasets, deep learning is often a reason why companies try to **aggregate data**: they need it to solve their particular problem.

### **1.2 Bringing the Model to the Data**

In 2017, Google published a paper and a blog post that changed this conversation considerably. Its proposal was that a dataset does not have to be centralized in order to train a model on it. The question it raised was the following:

> What if, instead of bringing the whole training corpus to one place, the model could be brought to the data, wherever that data is generated?

This is the idea of **federated learning**, a new and exciting subfield of machine learning, and it is the subject of this chapter.

### **1.3 Why This Reversal Matters**

The author highlights two consequences of this simple reversal:

1. **Individuals need not hand over their data.** To take part in the "deep learning supply chain", people do not technically have to send their data to anyone. Valuable models in healthcare, personal management, and other sensitive areas could be trained without anyone disclosing information about themselves. In theory, people could keep control of the only copy of their personal data, at least as far as deep learning is concerned.
2. **The competitive landscape changes.** Large enterprises that previously would not (or, for legal reasons, could not) share data about their customers could still earn revenue from that data. Some problem domains have been held back by the sensitivity of the data and by regulatory constraints. Healthcare is one example: its datasets are often locked up tightly, which makes research difficult.

---
## **2. Federated Learning**

**You do not have to have access to a dataset in order to learn from it.**

### **2.1 The Premise**

Many datasets contain information that is useful for solving important problems (the book's example is identifying cancer in an MRI scan), but it is hard to obtain such datasets in quantities large enough to train a sufficiently strong deep learning model. The main worry is that the same dataset that holds enough information to train the model also holds other information that presumably has nothing to do with the task, yet could harm someone if it were revealed.

**Federated learning** means that a model goes into a secure environment and learns to solve a problem there, without the data having to move anywhere. The chapter makes this concrete with an example.

### **2.2 Preprocessing the Email Data**

The first listing is the beginning of the preprocessing for the email example. What it does:

- It reads the lines of `spam.txt` and `ham.txt` (each line is one email) and turns every email into a `set` of its words, so repeated words and word order are lost.
- It collects all distinct words into a vocabulary (the special token `<unk>` is included from the start) and builds the dictionary `w2i`, which maps each word to its index.
- The function `to_indices` converts each email into a list of word indices of length `l=500`, padding with `<unk>` where an email has fewer words.

**What the code assumes.** This chapter's listings rely on files and classes that are not printed in the chapter itself:

| Needed by the code | What the book says about it |
|---|---|
| `spam.txt` and `ham.txt` | The Enron-Spam email dataset (the book's annotation points to `www2.aueb.gr/users/ion/data/enron-spam/`). The two files are available on the book's website and in its GitHub repository. |
| `Tensor`, `Embedding`, `MSELoss`, `SGD` | The text states that the embedding class comes from the deep learning framework created in Chapter 13. The other three names are used without being defined here and are assumed to belong to the same framework. |
| `phe` | A Python library for homomorphic encryption, installed with `pip install phe` (used from Section 7 on). |

The setup cells below, which are **not part of the book's code**, download the two text files and define the framework classes (copied from their final versions in the Chapter 13 notebook), so that this notebook can run on its own.

*Note on a repair:* the book prints the file name of the second `open` call with a typographic opening quote (`‘ham.txt'`), which is a syntax error. This notebook uses the plain quotes `'ham.txt'`.

*Note on the code:* all of the body of `to_indices` is indented inside the `if(len(line) < l)` test, so an email with 500 or more distinct words is simply left out of the result, whereas the surrounding text says that emails are made exactly 500 words long "by either trimming ... or padding". The notebook keeps the code as printed.

**Setup (not from the book).** The next cell downloads the two email files from the book's repository (they are large, roughly 11 MB and 21 MB).

In [ ]:
# Setup (not from the book): download the email files
!wget -q -nc https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/spam.txt
!wget -q -nc https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/ham.txt

**Setup (not from the book).** The next cell defines the framework classes that this chapter uses but does not print. It is the final version of the framework from the book's public repository (`iamtrask/Grokking-Deep-Learning`), which is the version that the code of Chapter 14 was written against. It contains `Tensor`, `SGD`, `Layer`, `Linear`, `Embedding`, `Tanh`, `Sigmoid`, `CrossEntropyLoss`, and `RNNCell`, plus `MSELoss` copied from the Chapter 13 notebook. Compared with the Chapter 13 listings, two things are extended: `Linear` accepts a `bias` option, and `Tensor` has a `softmax` method; the book's code calls both without printing them.

In [ ]:
# Setup (not from the book): framework classes of the book's repository
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None

        if(id is None):
            self.id = np.random.randint(0,1000000000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    return
                    print(self.id)
                    print(self.creation_op)
                    print(len(self.creators))
                    for c in self.creators:
                        print(c.creation_op)
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

                if(self.creation_op == "cross_entropy"):
                    dx = self.softmax_output - self.target_dist
                    self.creators[0].backward(Tensor(dx))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def softmax(self):
        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)
        return softmax_output

    def cross_entropy(self, target_indices):

        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)

        t = target_indices.data.flatten()
        p = softmax_output.reshape(len(t),-1)
        target_dist = np.eye(p.shape[1])[t]
        loss = -(np.log(p) * (target_dist)).sum(1).mean()

        if(self.autograd):
            out = Tensor(loss,
                         autograd=True,
                         creators=[self],
                         creation_op="cross_entropy")
            out.softmax_output = softmax_output
            out.target_dist = target_dist
            return out

        return Tensor(loss)


    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

class Layer(object):

    def __init__(self):
        self.parameters = list()

    def get_parameters(self):
        return self.parameters

class SGD(object):

    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha = alpha

    def zero(self):
        for p in self.parameters:
            p.grad.data *= 0

    def step(self, zero=True):

        for p in self.parameters:

            p.data -= p.grad.data * self.alpha

            if(zero):
                p.grad.data *= 0

class Linear(Layer):

    def __init__(self, n_inputs, n_outputs, bias=True):
        super().__init__()

        self.use_bias = bias

        W = np.random.randn(n_inputs, n_outputs) * np.sqrt(2.0/(n_inputs))
        self.weight = Tensor(W, autograd=True)
        if(self.use_bias):
            self.bias = Tensor(np.zeros(n_outputs), autograd=True)

        self.parameters.append(self.weight)

        if(self.use_bias):
            self.parameters.append(self.bias)

    def forward(self, input):
        if(self.use_bias):
            return input.mm(self.weight)+self.bias.expand(0,len(input.data))
        return input.mm(self.weight)

class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # this random initialiation style is just a convention from word2vec
        self.weight = Tensor((np.random.rand(vocab_size, dim) - 0.5) / dim, autograd=True)

        self.parameters.append(self.weight)

    def forward(self, input):
        return self.weight.index_select(input)

class MSELoss(Layer):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        return ((pred - target)*(pred - target)).sum(0)

class Tanh(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.tanh()

class Sigmoid(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.sigmoid()

class CrossEntropyLoss(object):

    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        return input.cross_entropy(target)

class RNNCell(Layer):

    def __init__(self, n_inputs, n_hidden, n_output, activation='sigmoid'):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        if(activation == 'sigmoid'):
            self.activation = Sigmoid()
        elif(activation == 'tanh'):
            self.activation == Tanh()
        else:
            raise Exception("Non-linearity not found")

        self.w_ih = Linear(n_inputs, n_hidden)
        self.w_hh = Linear(n_hidden, n_hidden)
        self.w_ho = Linear(n_hidden, n_output)

        self.parameters += self.w_ih.get_parameters()
        self.parameters += self.w_hh.get_parameters()
        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):
        from_prev_hidden = self.w_hh.forward(hidden)
        combined = self.w_ih.forward(input) + from_prev_hidden
        new_hidden = self.activation.forward(combined)
        output = self.w_ho.forward(new_hidden)
        return output, new_hidden

    def init_hidden(self, batch_size=1):
        return Tensor(np.zeros((batch_size,self.n_hidden)), autograd=True)

In [ ]:
import numpy as np
from collections import Counter
import random
import sys
import codecs
np.random.seed(12345)
with codecs.open('spam.txt',"r",encoding='utf-8',errors='ignore') as f:
    raw = f.readlines()

vocab, spam, ham = (set(["<unk>"]), list(), list())
for row in raw:
    spam.append(set(row[:-2].split(" ")))
    for word in spam[-1]:
        vocab.add(word)

with codecs.open('ham.txt',"r",encoding='utf-8',errors='ignore') as f:
    raw = f.readlines()

for row in raw:
    ham.append(set(row[:-2].split(" ")))
    for word in ham[-1]:
        vocab.add(word)

vocab, w2i = (list(vocab), {})
for i,w in enumerate(vocab):
    w2i[w] = i

def to_indices(input, l=500):
    indices = list()
    for line in input:
        if(len(line) < l):
            line = list(line) + ["<unk>"] * (l - len(line))
            idxs = list()
            for word in line:
                idxs.append(w2i[word])
            indices.append(idxs)
    return indices

---
## **3. Learning to Detect Spam**

**Suppose you want to train a model across many people's emails to detect spam.**

### **3.1 The Enron Dataset**

The use case is **email classification**. The first model is trained on the publicly available **Enron dataset**, a large corpus of emails that were released as a consequence of the well-known Enron lawsuit and that has since become a standard corpus for email analytics. The author mentions that he once knew someone who read and annotated this dataset professionally, and that people had emailed each other all sorts of unusual (and often very personal) things. Because everything was made public in the court case, the data is free to use.

The code of the previous section and of this one is only **preprocessing**. Its purpose is to prepare the emails for forward propagation into the embedding class of the Chapter 13 framework. As before, the words of the corpus are turned into lists of indices, and every email is made exactly 500 words long, either by trimming or by padding with `<unk>` tokens. This makes the final dataset "square" (all rows have the same length).

### **3.2 Train and Test Splits**

The next cell converts the emails into index lists and then builds the training and test sets:

- The **last 1,000 spam emails and the last 1,000 ham emails** form the test set; everything before them forms the training set.
- The two classes are **interleaved**: a spam email (target `[1]`) is followed by a ham email (target `[0]`). The index `i%len(...)` wraps around, so the shorter class would be cycled if the two classes had different lengths. In this way both datasets stay balanced.

In [ ]:
spam_idx = to_indices(spam)
ham_idx = to_indices(ham)

train_spam_idx = spam_idx[0:-1000]
train_ham_idx = ham_idx[0:-1000]

test_spam_idx = spam_idx[-1000:]
test_ham_idx = ham_idx[-1000:]

train_data = list()
train_target = list()

test_data = list()
test_target = list()

for i in range(max(len(train_spam_idx),len(train_ham_idx))):
    train_data.append(train_spam_idx[i%len(train_spam_idx)])
    train_target.append([1])

    train_data.append(train_ham_idx[i%len(train_ham_idx)])
    train_target.append([0])

for i in range(max(len(test_spam_idx),len(test_ham_idx))):
    test_data.append(test_spam_idx[i%len(test_spam_idx)])
    test_target.append([1])

    test_data.append(test_ham_idx[i%len(test_ham_idx)])
    test_target.append([0])

### **3.3 The `train()` and `test()` Functions**

The next cell defines two functions.

- `train()` processes the data in batches (`batch_size=500` by default). For every batch it first sets the weight of the padding token `<unk>` back to zero. It then runs the forward pass `model.forward(input).sum(1).sigmoid()` (look up the embedding of each word, add the values of the words of an email, and squash the sum with a sigmoid), computes the loss with the global `criterion`, back-propagates, and takes a step with the optimizer. The loss of each batch is divided by the batch size and averaged, and the running value is printed.
- `test()` performs the same forward pass on the test data and returns the fraction of emails for which `pred > 0.5` agrees with the target.

*Note on repairs:* the book's version of `train()` cannot run as printed, and it also does not behave as the later sections describe, for two reasons.

1. The book slices the batches with the name `bs` and divides the loss by `bs`, but `bs` is never defined; the function's parameter is called `batch_size`. This notebook replaces `bs` by `batch_size` in these three places.
2. `train()` uses the global `optim`, which was created for one particular model. Later sections call `train()` on a **copy** of the model (`copy.deepcopy(model)`), and the global optimizer would keep updating the original model's weights instead of the copy's, so that the copies would never change. To make each call update the model it receives, this notebook adds one line at the top of `train()`, `optim = SGD(parameters=model.get_parameters(), alpha=0.01)`. It is the same line that the book prints in the next cell, only repeated inside the function so that the optimizer is bound to the model passed in. The global `criterion` is used as printed.

Nothing else was changed (the comment `# padding token should stay at 0` is the book's).

In [ ]:
def train(model, input_data, target_data, batch_size=500, iterations=5):
    optim = SGD(parameters=model.get_parameters(), alpha=0.01)
    n_batches = int(len(input_data) / batch_size)
    for iter in range(iterations):
        iter_loss = 0
        for b_i in range(n_batches):

            # padding token should stay at 0
            model.weight.data[w2i['<unk>']] *= 0
            input = Tensor(input_data[b_i*batch_size:(b_i+1)*batch_size], autograd=True)
            target = Tensor(target_data[b_i*batch_size:(b_i+1)*batch_size], autograd=True)

            pred = model.forward(input).sum(1).sigmoid()
            loss = criterion.forward(pred,target)
            loss.backward()
            optim.step()

            iter_loss += loss.data[0] / batch_size

            sys.stdout.write("\r\tLoss:" + str(iter_loss / (b_i+1)))
        print()
    return model


def test(model, test_input, test_output):

    model.weight.data[w2i['<unk>']] *= 0

    input = Tensor(test_input, autograd=True)
    target = Tensor(test_output, autograd=True)

    pred = model.forward(input).sum(1).sigmoid()
    return ((pred.data > 0.5) == target.data).mean()

### **3.4 Training Without Federation**

With these functions, a network can be initialized and trained with a few lines. The model is an `Embedding` with `dim=1`, that is, **one weight per word of the vocabulary**, all starting at zero; the loss is `MSELoss`, and the optimizer is `SGD` with `alpha=0.01`. The book prints the model setup and the training loop as one listing; they are split here into two cells so that the zero-initialized model can be recreated on its own (this matters in Section 4).

In [ ]:
model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0
criterion = MSELoss()
optim = SGD(parameters=model.get_parameters(), alpha=0.01)

In [ ]:
for i in range(3):
    model = train(model, train_data, train_target, iterations=1)
    print("% Correct on Test Set: " + \
          str(test(model, test_data, test_target)*100))

	Loss:0.037140416860871446
% Correct on Test Set: 98.65
	Loss:0.011258669226059108
% Correct on Test Set: 99.15
	Loss:0.008068268387986223
% Correct on Test Set: 99.45


After only three iterations the network classifies the test set with **99.45 %** accuracy. The test set is balanced, so this is quite a good result. The book reports the following values:

| Iteration | Training loss | % correct on the test set |
|---|---|---|
| 1 | 0.03714 | 98.65 |
| 2 | 0.01126 | 99.15 |
| 3 | 0.00807 | 99.45 |

(The exact numbers you obtain depend on your copy of the data and on the framework implementation.)

---
## **4. Let's Make It Federated**

**The previous example was plain vanilla deep learning. Let's protect privacy.**

### **4.1 Simulating Several Owners of Data**

In the previous section all emails were in one place, which the author calls the old-school way of doing things (and one that is still far too common). To simulate a **federated learning environment**, the training emails are now divided into separate collections, one for each of three people: Bob gets the first 1,000 training examples, Alice the next 1,000, and Sue the remaining ones.

In [ ]:
bob = (train_data[0:1000], train_target[0:1000])
alice = (train_data[1000:2000], train_target[1000:2000])
sue = (train_data[2000:], train_target[2000:])

### **4.2 Training Across the Three Collections**

Now the same training as before is performed across each person's collection of emails. In every round the following happens:

1. **Step 1:** a copy of the model is sent to Bob, who trains it for one iteration on his own emails.
2. **Step 2:** a copy of the model is sent to Alice, who does the same on her emails.
3. **Step 3:** a copy of the model is sent to Sue, who does the same on hers.
4. **Averaging:** the weights of the three returned models are averaged, and the averaged model is evaluated on the test set.

The author points out that some federated learning methods aggregate after every batch (or every group of batches); here he keeps it simple and averages once per iteration.

*Note on a repair:* the loop uses `copy.deepcopy`, but the book imports `copy` only later, in the next section, so the cell would fail with a `NameError` when run in order. This notebook adds the line `import copy` (the same line that the book prints in Section 5) at the top of this cell.

*Note on the starting model:* as printed, the loop starts from the `model` object that is currently in memory. If you run the notebook straight through, that is the model already trained in Section 3, and the numbers will differ from the book's excerpt, whose first-round losses (see below) are those of a model that starts from zero. To reproduce the book's setting, run the *model setup* cell of Section 3.4 again (it creates a fresh model with all weights at zero) before running the cell below.

In [ ]:
import copy

for i in range(3):
    print("Starting Training Round...")
    print("\tStep 1: send the model to Bob")
    bob_model = train(copy.deepcopy(model), bob[0], bob[1], iterations=1)

    print("\n\tStep 2: send the model to Alice")
    alice_model = train(copy.deepcopy(model),
                        alice[0], alice[1], iterations=1)

    print("\n\tStep 3: Send the model to Sue")
    sue_model = train(copy.deepcopy(model), sue[0], sue[1], iterations=1)

    print("\n\tAverage Everyone's New Models")
    model.weight.data = (bob_model.weight.data + \
                         alice_model.weight.data + \
                         sue_model.weight.data)/3

    print("\t% Correct on Test Set: " + \
          str(test(model, test_data, test_target)*100))

    print("\nRepeat!!\n")

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.007924345075144994

	Step 2: send the model to Alice
	Loss:0.007990960300900699

	Step 3: Send the model to Sue
	Loss:0.006388663983005001

	Average Everyone's New Models
	% Correct on Test Set: 99.5

Repeat!!

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.007331416633817908

	Step 2: send the model to Alice
	Loss:0.00729249465902758

	Step 3: Send the model to Sue
	Loss:0.0060201301162778244

	Average Everyone's New Models
	% Correct on Test Set: 99.6

Repeat!!

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.006884940668687031

	Step 2: send the model to Alice
	Loss:0.006776291525625331

	Step 3: Send the model to Sue
	Loss:0.005694468896433619

	Average Everyone's New Models
	% Correct on Test Set: 99.65

Repeat!!



The excerpt of the output printed in the book shows a loss of about 0.219 for Bob in Step 1 and about 0.0154 for Sue in Step 3, and it ends with **98.8 %** correct on the test set after averaging. The federated model therefore learns to nearly the same performance as the centralized one, and, at least in theory, nobody ever had access to the training data.

Or did they? Each person changes the model in some way, after all. **Can one really discover nothing about their dataset from these changes?** The next section addresses this question.

---
## **5. Hacking into Federated Learning**

**A toy example shows how the training data can still be learned.**

### **5.1 Two Big Challenges: Performance and Privacy**

Federated learning faces two major challenges, **performance** and **privacy**, and both are at their worst when each participant has only a handful of training examples. If someone has only a few examples, or if the improvement they send back is computed from only a few examples (a single training batch), then you can still learn quite a lot about their data. As for performance: with 10,000 people who each hold a little data, you would spend most of the time sending the model back and forth and very little time training, especially for a large model.

### **5.2 What a Single Batch Reveals**

To see what can be learned from the update of one user over a single batch, the book builds a toy example. Bob has an email in his inbox containing his password: "My computer password is pizza". The email is converted to word indices, its target is `0` (not spam), and a freshly created model with zero weights is trained for one iteration with `batch_size=1`.

Notes on this cell: it creates a **new** zero-initialized `model`, replacing the one from the previous sections, and `bobs_model` is the trained copy. The book prints `import copy` here, as the first line of this listing. Because of the optimizer repair described in Section 3.3, the call to `train()` updates the copy that is passed to it.

In [ ]:
import copy

bobs_email = ["my", "computer", "password", "is", "pizza"]

bob_input = np.array([[w2i[x] for x in bobs_email]])
bob_target = np.array([[0]])

model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

bobs_model = train(copy.deepcopy(model),
                   bob_input, bob_target, iterations=1, batch_size=1)

	Loss:0.25


### **5.3 Reading the Vocabulary from the Weight Update**

Anyone who sees the model before and after Bob's update can compare the weights. The next cell prints the words whose weights changed. The model is a lookup table with one weight per word, so only the entries of the words that occurred in the batch can have been touched (this is the reason why the changed weights give away the vocabulary).

In [ ]:
for i, v in enumerate(bobs_model.weight.data - model.weight.data):
    if(v != 0):
        print(vocab[i])

pizza
my
password
computer
is


The book's output lists the five words **is, pizza, computer, password, my**, in this order (the order of the words follows the order of the vocabulary list, and so it can differ between runs). By looking at which weights changed, one learns the vocabulary of Bob's email and can infer its meaning: his secret password, and probably his favorite food as well.

This raises a question: **how can federated learning be used at all if the training data can be read so easily from a weight update?** The following sections give the chapter's answer.

---
## **6. Secure Aggregation**

**Average the weight updates of zillions of people before anyone can see them.**

### **6.1 Randomized Response**

The solution is never to let Bob put his gradient out in the open. But then how can Bob contribute his gradient if nobody should see it? The social sciences use an interesting technique called **randomized response**.

Suppose you run a survey and want to ask 100 people whether they have committed a heinous crime. Everybody would answer "No", even if you promised not to tell. The technique therefore changes the procedure:

1. Each person flips a coin **twice**, somewhere you cannot see.
2. If the **first** flip is heads, the person answers **honestly**.
3. If the first flip is tails, the person answers "Yes" or "No" according to the **second** flip.

You never really ask anybody whether they committed a crime, and the true answers are hidden in the random noise of the two coin flips. According to the book, if 60 % of the respondents say "Yes", simple math shows that about 70 % of the people surveyed committed the crime (give or take a few percentage points). A quick check, assuming fair coins: half of the respondents answer honestly and half answer at random with a 50 % chance of "Yes", so the share of "Yes" answers is 0.5 p + 0.25, where p is the true share; setting this equal to 0.6 gives p = 0.7.

The random noise makes it plausible that whatever you learn about one person came from the noise and not from that person.

> **Privacy via plausible deniability.** The chance that a particular answer came from random noise rather than from the individual protects that individual's privacy by giving them *plausible deniability*. This is the basis of secure aggregation and, more generally, of much of **differential privacy**.

### **6.2 Aggregate Statistics Only**

In such a survey you look only at **aggregate statistics**: you never see anyone's answer directly, only pairs of answers or perhaps larger groupings. Consequently, the more people you can aggregate before adding noise, the less noise is needed to hide them, and the more accurate the findings are.

### **6.3 Secure Aggregation for Federated Learning**

In federated learning one could add a large amount of noise to every update, but this would hurt training. Instead, the plan is to **first sum the gradients of all participants in such a way that nobody can see anyone's gradient except their own**. The class of problems concerned with doing this is called **secure aggregation**, and one more tool is needed to solve it: **homomorphic encryption**.

---
## **7. Homomorphic Encryption**

**You can perform arithmetic on encrypted values.**

### **7.1 The Idea**

One of the most exciting frontiers of research lies at the intersection of artificial intelligence (including deep learning) and cryptography, and homomorphic encryption is front and center there. Loosely stated, **homomorphic encryption** allows you to perform computations on encrypted values without decrypting them. Here the interest is in performing **addition** on such values. A full explanation would fill a book of its own, so the chapter only introduces a few definitions:

| Term | Meaning |
|---|---|
| **public key** | lets you *encrypt* numbers |
| **private key** | lets you *decrypt* encrypted numbers |
| **ciphertext** | an encrypted value |
| **plaintext** | an unencrypted value |

### **7.2 An Example with the `phe` Library**

The example uses the `phe` library. To install it, run the following command (in Google Colab, the command is run from a code cell with an exclamation mark in front of it, as in the setup cell below, which is **not part of the book's code**):

```
pip install phe
```

The cell below creates a key pair with `phe.generate_paillier_keypair(n_length=1024)` and then proceeds in four steps, which the book annotates as follows: `x = public_key.encrypt(5)` encrypts the number 5; `y = public_key.encrypt(3)` encrypts the number 3; `z = x + y` adds the two encrypted values; and `z_ = private_key.decrypt(z)` decrypts the result.

**Setup (not from the book).** The next cell installs the `phe` library.

In [ ]:
# Setup (not from the book): install the library
!pip install -q phe

In [ ]:
import phe

public_key, private_key = phe.generate_paillier_keypair(n_length=1024)

x = public_key.encrypt(5)

y = public_key.encrypt(3)

z = x + y

z_ = private_key.decrypt(z)
print("The Answer: " + str(z_))

The Answer: 8


The program prints `The Answer: 8`. Two numbers were encrypted and added while they were still encrypted, and only the final result was decrypted.

The chapter also mentions a technique that is a sort of cousin of homomorphic encryption, **secure multi-party computation**, and refers the reader to the "Cryptography and Machine Learning" blog by Morten Dahl.

### **7.3 Returning to Secure Aggregation**

Since numbers that cannot be seen can be added together, the solution to secure aggregation becomes plain:

1. The person who initializes the model sends a `public_key` to Bob, Alice, and Sue, so that each of them can **encrypt** their weight update.
2. Bob, Alice, and Sue, who do **not** have the private key, communicate directly with each other and **accumulate** all their gradients into a single final update.
3. This single update is sent back to the model owner, who **decrypts** it with the `private_key`.

---
## **8. Homomorphically Encrypted Federated Learning**

**Use homomorphic encryption to protect the gradients that are being aggregated.**

### **8.1 The Training Scheme**

The listing below combines the federated training of Section 4 with the encryption of Section 7. The new function `train_and_encrypt()` trains a copy of the model on one person's data and then encrypts every weight of the result with the public key, returning a NumPy array of encrypted numbers. In each round the following happens:

1. **Steps 1 to 3:** the model is sent to Bob, to Alice, and to Sue, and each returns an *encrypted* model.
2. **Step 4:** the three encrypted models are added to each other (`aggregated_model`). Addition works directly on the encrypted values.
3. **Step 5:** only the aggregated model is sent back to the model owner, who can decrypt it with the private key. The decrypted values are divided by 3, and the result becomes the new model, which is then evaluated on the test set.

Notes on this cell:

- It creates a new zero-initialized `model`, and it uses `phe` and `copy`, which were imported in the previous sections.
- The book's comment in the code states that in production `n_length` should be at least 1024; the cell uses `n_length=128`, whereas the earlier demonstration used 1024.
- The function receives the key as the argument `pubkey` but encrypts with the global `public_key`, which holds the same key.
- *Book inconsistency:* in Step 5, the printed loop decrypts `sue_encrypted_model` and not `aggregated_model`, so the sum built in Step 4 is never used, although the text describes the decryption of the aggregated model. Using `aggregated_model` there would match the description. The notebook keeps the code exactly as printed.

In [ ]:
model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

# note that in production the n_length should be at least 1024
public_key, private_key = phe.generate_paillier_keypair(n_length=128)

def train_and_encrypt(model, input, target, pubkey):
    new_model = train(copy.deepcopy(model), input, target, iterations=1)

    encrypted_weights = list()
    for val in new_model.weight.data[:,0]:
        encrypted_weights.append(public_key.encrypt(val))
    ew = np.array(encrypted_weights).reshape(new_model.weight.data.shape)

    return ew

for i in range(3):
    print("\nStarting Training Round...")
    print("\tStep 1: send the model to Bob")
    bob_encrypted_model = train_and_encrypt(copy.deepcopy(model),
                                            bob[0], bob[1], public_key)

    print("\n\tStep 2: send the model to Alice")
    alice_encrypted_model=train_and_encrypt(copy.deepcopy(model),
                                            alice[0],alice[1],public_key)

    print("\n\tStep 3: Send the model to Sue")
    sue_encrypted_model = train_and_encrypt(copy.deepcopy(model),
                                            sue[0], sue[1], public_key)

    print("\n\tStep 4: Bob, Alice, and Sue send their")
    print("\tencrypted models to each other.")
    aggregated_model = bob_encrypted_model + \
                       alice_encrypted_model + \
                       sue_encrypted_model

    print("\n\tStep 5: only the aggregated model")
    print("\tis sent back to the model owner who")
    print("\t can decrypt it.")
    raw_values = list()
    for val in sue_encrypted_model.flatten():
        raw_values.append(private_key.decrypt(val))
    new = np.array(raw_values).reshape(model.weight.data.shape)/3
    model.weight.data = new

    print("\t% Correct on Test Set: " + \
              str(test(model, test_data, test_target)*100))


Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.21908166249699718

	Step 2: send the model to Alice
	Loss:0.2937106899184867

	Step 3: Send the model to Sue
	Loss:0.033339966977175894

	Step 4: Bob, Alice, and Sue send their
	encrypted models to each other.

	Step 5: only the aggregated model
	is sent back to the model owner who
	 can decrypt it.
	% Correct on Test Set: 98.75

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.06366414053035604

	Step 2: send the model to Alice
	Loss:0.06100035791351306

	Step 3: Send the model to Sue
	Loss:0.02548392041662726

	Step 4: Bob, Alice, and Sue send their
	encrypted models to each other.

	Step 5: only the aggregated model
	is sent back to the model owner who
	 can decrypt it.
	% Correct on Test Set: 99.05000000000001

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.058477976535441636

	Step 2: send the model to Alice
	Loss:0.05987976552444443

	Step 3: Send the model to Sue
	Loss:0.024763

### **8.2 What the Scheme Achieves**

With this scheme there is an added step: Alice, Bob, and Sue add up their homomorphically encrypted models before anything is sent back to the model owner, so the owner never sees which update came from which person. This is a form of **plausible deniability**. In production, the author adds, one would also include additional random noise, enough to meet the privacy threshold that Bob, Alice, and Sue require according to their personal preferences; this is left for future work.

The output printed in the book shows the same kind of messages as before (for example, a loss of about 0.219 for Bob in Step 1 and about 0.294 for Alice in Step 2) and ends with **99.15 %** correct on the test set. Hence the model still learns the task while the individual updates stay encrypted.

---
## **9. Summary**

**Federated learning is one of the most exciting breakthroughs in deep learning.**

The key points of Chapter 15 are:

- Deep learning needs training data, and the most valuable data is often the most personal. **Federated learning** reverses the usual flow: instead of bringing the data to the model, the **model is brought to the data**, so people do not have to send their data to anyone, and organizations that cannot share data (for legal or competitive reasons) can still contribute to training.
- A federated version of the **spam classifier** (one embedding weight per word, trained on Bob's, Alice's, and Sue's emails separately and then averaged) reached nearly the same test accuracy as the centralized model: 98.8 % in the book's excerpt, compared with 99.45 % for the centralized model.
- Federated learning has two big challenges, **performance** (too much time spent sending models back and forth when people hold little data) and **privacy**. A weight update computed from a single email reveals which words occur in it, and in the toy example it exposes Bob's password.
- **Randomized response** shows how noise gives **plausible deniability**: the more people are aggregated before noise is added, the less noise is needed and the more accurate the result. For federated learning, the plan is to sum the gradients of all participants so that nobody sees anyone else's gradient. This is called **secure aggregation**.
- **Homomorphic encryption** allows the addition of encrypted numbers: a public key encrypts, a private key decrypts, and the sum of ciphertexts decrypts to the sum of the plaintexts (5 + 3 = 8 in the example with `phe`).
- In the **homomorphically encrypted federated learning** scheme, each participant encrypts the trained weights, the encrypted models are added together, and the model owner decrypts only the aggregate. The test accuracy in the book's output is 99.15 %. In production, additional random noise would be added to reach the privacy level that the participants require.
- The author believes that federated learning will change the landscape of deep learning, because it will unlock datasets that were previously too sensitive to use, with great social benefit and entrepreneurial opportunity. It belongs to a wider convergence of encryption and artificial intelligence research, which he considers the most exciting convergence of the decade.
- What holds these techniques back is that modern deep learning toolkits do not offer them. The tipping point will come when anyone can run `pip install ...` and obtain frameworks in which privacy and security are first-class citizens, with federated learning, homomorphic encryption, differential privacy, and secure multi-party computation built in, usable without being an expert. The author works on this goal with open-source volunteers in the **OpenMined** project, which extends major deep learning frameworks with these primitives, and he invites readers to support or join it.